# 04 · 결과 집계

팀원 전체의 원장을 합쳐 **판정별로** 정리하고, 그대로 붙여넣어 보고할 마크다운을 만든다.

In [ ]:
# ── 부트스트랩 · 이 셀을 가장 먼저 실행 ──────────────────────────
# 집계 노트북은 원장 CSV 만 읽는다. 1.1GB 데이터도 비밀번호도 필요 없다.
RUNNER_NAME = "본인이름"        # ← 여기만 바꾼다

import os, sys, subprocess
if not os.path.exists('/content/lga-repo/src/config.py'):
    subprocess.run(['git','clone','-q',
                    'https://github.com/hyunku9566/lga_data.git','/content/lga-repo'])
else:
    subprocess.run(['git','-C','/content/lga-repo','pull','-q'])

for m in [k for k in list(sys.modules)
          if k.startswith('src') or k in ('config','lib_lga','experiment','download','bootstrap')]:
    del sys.modules[m]
for _p in ('/content/lga-repo/src', '/content/lga-repo'):
    if _p not in sys.path: sys.path.insert(0, _p)

from bootstrap import setup
C = setup(runner=RUNNER_NAME, need_data=False)     # ← 데이터 안 받는다
import experiment as E

# 한글 폰트 (없으면 그래프 글자가 깨진다)
try:
    import matplotlib, matplotlib.pyplot as plt
    if not any('Nanum' in f.name for f in matplotlib.font_manager.fontManager.ttflist):
        subprocess.run(['apt-get','-qq','install','-y','fonts-nanum'], capture_output=True)
        matplotlib.font_manager._load_fontmanager(try_read_cache=False)
    plt.rcParams['font.family'] = 'NanumGothic'
    plt.rcParams['axes.unicode_minus'] = False
except Exception as e:
    print(f'한글 폰트 설정 실패 (그래프 글자가 깨질 수 있다): {e}')


In [ ]:
if 'E' not in globals(): exec(open('/content/lga-repo/colab/_ensure.py').read())

import pandas as pd, numpy as np, sync
sync.pull_ledgers()                                   # 저장소에서 팀 전체 원장 받기
led = E.read_all_ledgers(sync.repo_ledger_dir())      # Drive 아닌 저장소를 읽는다
print(f'총 {len(led)}건, 실행자 {led.runner.nunique() if len(led) else 0}명')
if len(led):
    print(led.groupby('runner').size().to_string())
else:
    print('원장이 비어 있다. 팀원이 00_runner 로 실험을 돌리고 푸시해야 쌓인다.')


## 헤드라인

In [ ]:
if 'led' not in globals():
    if 'E' not in globals(): exec(open('/content/lga-repo/colab/_ensure.py').read())
    led = E.read_all_ledgers()
if not len(led):
    print('원장이 비어 있다. 팀원이 03 노트북에서 실험을 돌려야 집계할 게 생긴다.')

if len(led):
    vc = led.verdict.value_counts()
    n_cand = int(vc.get('채택후보', 0))
    print(f'채택 후보 {n_cand}건 / 보류 {vc.get("보류",0)} / 노이즈 {vc.get("노이즈",0)} / 기각 {vc.get("기각",0)}')
    print()
    if n_cand:
        print('※ 채택 후보라도 LB 개선은 보장되지 않는다.')
        print('  폴드2024/2023 HPO 순위 Spearman -0.806, 27개 중 두 폴드 동시 개선 0개였다.')
    else:
        print('※ 채택 후보 없음. 두 폴드가 서로 역전되는 관계라 이게 정상적인 결과다.')

## 판정별 표

In [ ]:
if 'led' not in globals():
    if 'E' not in globals(): exec(open('/content/lga-repo/colab/_ensure.py').read())
    led = E.read_all_ledgers()
if not len(led):
    print('원장이 비어 있다. 팀원이 03 노트북에서 실험을 돌려야 집계할 게 생긴다.')

COLS = ['runner','name','params_json','m24','delta24','se24','m23','delta23','se23','lb_lo','lb_hi','seeds']
for v in ['채택후보','보류','노이즈','기각']:
    sub = led[led.verdict == v] if len(led) else led
    print(f'\n{"="*70}\n{v}  ({len(sub)}건)\n{"="*70}')
    if len(sub):
        display(sub.sort_values('delta24', ascending=False)[COLS].head(20))

## 두 폴드 산점도

**두 폴드가 실제로 얼마나 어긋나는지 눈으로 확인한다.**
1사분면(오른쪽 위)에 있는 점만 채택 후보다. 과거 27개 HPO 조합에서는 그 영역이 **비어 있었다.**

In [ ]:
if 'led' not in globals():
    if 'E' not in globals(): exec(open('/content/lga-repo/colab/_ensure.py').read())
    led = E.read_all_ledgers()
if not len(led):
    print('원장이 비어 있다. 팀원이 03 노트북에서 실험을 돌려야 집계할 게 생긴다.')

import matplotlib.pyplot as plt
if len(led):
    fig, ax = plt.subplots(figsize=(7, 6.5))
    colors = {'채택후보':'#2E7D32','보류':'#F9A825','노이즈':'#9E9E9E','기각':'#C62828'}
    for v, g in led.groupby('verdict'):
        ax.scatter(g.delta24, g.delta23, s=55, alpha=.8,
                   c=colors.get(v, '#666'), label=f'{v} ({len(g)})', edgecolors='white', linewidths=.6)
    s24 = 2*led.se24.median(); s23 = 2*led.se23.median()
    ax.axhline(0, c='#333', lw=.9); ax.axvline(0, c='#333', lw=.9)
    ax.axhspan(-s23, s23, color='#999', alpha=.10)
    ax.axvspan(-s24, s24, color='#999', alpha=.10)
    ax.set_xlabel('폴드2024 효과 (기준선 대비)'); ax.set_ylabel('폴드2023 효과')
    ax.set_title('두 폴드 효과 비교 — 회색 띠 안은 노이즈')
    ax.legend(frameon=False)
    if len(led) > 2:
        r = led[['delta24','delta23']].corr(method='spearman').iloc[0,1]
        ax.text(.02,.02, f'Spearman {r:+.3f}', transform=ax.transAxes, fontsize=11)
    plt.tight_layout(); plt.show()

## 보고용 마크다운 (그대로 복사)

In [ ]:
if 'led' not in globals():
    if 'E' not in globals(): exec(open('/content/lga-repo/colab/_ensure.py').read())
    led = E.read_all_ledgers()
if not len(led):
    print('원장이 비어 있다. 팀원이 03 노트북에서 실험을 돌려야 집계할 게 생긴다.')

if len(led):
    lines = ['## 실험 집계', '',
             f'- 총 {len(led)}건 (실행자 {led.runner.nunique()}명)',
             f'- 채택 후보 **{int((led.verdict=="채택후보").sum())}건** / '
             f'보류 {int((led.verdict=="보류").sum())} / '
             f'노이즈 {int((led.verdict=="노이즈").sum())} / '
             f'기각 {int((led.verdict=="기각").sum())}', '']
    cand = led[led.verdict=='채택후보'].sort_values('delta24', ascending=False)
    if len(cand):
        lines += ['### 채택 후보', '',
                  '| 아이디어 | 설정 | 폴드2024 | 폴드2023 | LB 기대 | 실행자 |',
                  '|---|---|---|---|---|---|']
        for _, r in cand.head(10).iterrows():
            lines.append(f'| {r["name"]} | `{r["params_json"]}` | {r["delta24"]:+.1f} '
                         f'| {r["delta23"]:+.1f} | {r["lb_lo"]:+.1f} ~ {r["lb_hi"]:+.1f} | {r["runner"]} |')
        lines.append('')
    hold = led[led.verdict=='보류'].sort_values('delta24', ascending=False)
    if len(hold):
        lines += ['### 보류 (폴드 불일치)', '']
        for _, r in hold.head(8).iterrows():
            lines.append(f'- {r["name"]} `{r["params_json"]}` — 2024 {r["delta24"]:+.1f} / '
                         f'2023 {r["delta23"]:+.1f}')
        lines.append('')
    lines += ['### 주의', '',
              '- 채택 후보라도 LB 개선은 보장되지 않는다. 폴드2024/2023 HPO 순위는 Spearman −0.806.',
              '- LB 기대범위는 실측 전이율 0.06~0.5배를 곱한 값이다.',
              '- **블렌드 가중치는 이 표로 판단하면 안 된다** (CV +14.2 → LB −11.66 사례).']
    print('\n'.join(lines))